In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv


In [2]:
!pip uninstall -y torch torchvision torchaudio torchao transformers peft unsloth unsloth-zoo


!pip install --no-cache-dir torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --no-cache-dir transformers==4.46.3 peft accelerate bitsandbytes trl datasets

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 272.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 73.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "unsloth/meta-llama-3.1-8b-bnb-4bit"

try:
    
    bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=True,       
)

    
    print("⏳ মডেল লোড হচ্ছে, দয়া করে অপেক্ষা করুন...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": 0}
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    print("✅ অভিনন্দন! মডেল এবং লাইব্রেরি সফলভাবে লোড হয়েছে।")
    
except Exception as e:
    print(f"❌ এখনো সমস্যা হচ্ছে: {e}")


⏳ মডেল লোড হচ্ছে, দয়া করে অপেক্ষা করুন...


config.json: 0.00B [00:00, ?B/s]

2026-04-01 20:15:12.533625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775074512.750075      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775074512.809971      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775074513.261526      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775074513.261566      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775074513.261569      24 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

✅ অভিনন্দন! মডেল এবং লাইব্রেরি সফলভাবে লোড হয়েছে।


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import gc
import pandas as pd
import sqlite3
from datetime import datetime
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer


gc.collect()
torch.cuda.empty_cache()

In [5]:
class DatasetProcessor:
    def __init__(self, path):
        self.path = path
    def load_and_preprocess(self):
        df = pd.read_csv(self.path).dropna().head(200)
        dataset = Dataset.from_pandas(df)
        def formatting_prompts_func(example):
            u_prompt = example['Questions']
            a_response = example['Answers']
            text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{u_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{a_response}<|eot_id|>"
            return {"text": text}
        return dataset.map(formatting_prompts_func)

class LLAMAFineTuner:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
    def train(self, dataset, sft_config, lora_config):
        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, lora_config)
        trainer = SFTTrainer(
            model=self.model,
            train_dataset=dataset,
            args=sft_config,
            processing_class=self.tokenizer
        )
        print("🚀 ফাইন-টুনিং শুরু হচ্ছে...")
        return trainer.train(), self.model



In [6]:

!pip uninstall -y transformers evaluate
!pip install --no-cache-dir transformers==4.46.3 evaluate rouge_score absl-py

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 222.2 MB/s eta 0:00:00


In [7]:
import sqlite3
import json
from datetime import datetime
import evaluate 

class Evaluator:
    def __init__(self, db_name="assessment_results.db"):
        self.db_name = db_name
        try:
            self.bleu = evaluate.load("bleu")
            self.rouge = evaluate.load("rouge")
        except:
            print("⚠️ মেট্রিক্স লোড করা যায়নি, দয়া করে !pip install evaluate rouge_score রান করুন।")
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_name) as conn:
            # Task 2 Requirement: LLAMAExperiments Table
            conn.execute("""CREATE TABLE IF NOT EXISTS LLAMAExperiments 
                (id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, 
                lora_config TEXT, train_loss REAL, metrics TEXT, timestamp TEXT)""")
            
            # Task 2 Requirement: GeneratedResponses Table
            conn.execute("""CREATE TABLE IF NOT EXISTS GeneratedResponses 
                (id INTEGER PRIMARY KEY AUTOINCREMENT, experiment_id INTEGER, 
                input_text TEXT, response_text TEXT, timestamp TEXT)""")

    def calculate_metrics(self, predictions, references):
        b_res = self.bleu.compute(predictions=predictions, references=references)
        r_res = self.rouge.compute(predictions=predictions, references=references)
        return {"bleu": round(b_res['bleu'], 4), "rougeL": round(r_res['rougeL'], 4)}

    
    def log_results(self, loss, metrics, prompt, response):
        with sqlite3.connect(self.db_name) as conn:
            cursor = conn.cursor()
            # lore config and matics save 
            cursor.execute("INSERT INTO LLAMAExperiments (model_name, train_loss, metrics, timestamp) VALUES (?,?,?,?)",
                         ("Llama-3.1-8B-bnb-4bit", loss, json.dumps(metrics), datetime.now().isoformat()))
            
            exp_id = cursor.lastrowid
            # input & output save
            cursor.execute("INSERT INTO GeneratedResponses (experiment_id, input_text, response_text, timestamp) VALUES (?,?,?,?)",
                         (exp_id, prompt, response, datetime.now().isoformat()))
            conn.commit()

In [8]:

#1.Model Tokenizer load 

model_id = "unsloth/meta-llama-3.1-8b-bnb-4bit"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0} 
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 2. configuration setup
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

sft_config = SFTConfig(
    output_dir="./outputs",
    max_seq_length=256,            
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=4, 
    max_steps=40,                  
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,   
    report_to="none"
)

# 3. Run
DATA_PATH = "/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv"
processor = DatasetProcessor(DATA_PATH)
dataset = processor.load_and_preprocess()

tuner = LLAMAFineTuner(model, tokenizer)
train_result, fine_tuned_model = tuner.train(dataset, sft_config, lora_config)

# 1.Generation 
test_prompt = "আমার খুব একা লাগছে, আমি কি করতে পারি?"
inputs = tokenizer(f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{test_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", return_tensors="pt").to("cuda")

outputs = fine_tuned_model.generate(**inputs, max_new_tokens=64)
# 'response' 
response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

# 2.Evaluation 
evaluator = Evaluator()
test_reference = ["আমি বুঝতে পারছি আপনার একা লাগছে, আপনি চাইলে আপনার প্রিয় কারো সাথে কথা বলতে পারেন।"] 
metrics = evaluator.calculate_metrics([response], [test_reference])

# 3.save database
evaluator.log_results(train_result.training_loss, metrics, test_prompt, response)

print(f"✅ সম্পন্ন হয়েছে! এআই উত্তর: {response}")
print(f"📊 মেট্রিক্স (BLEU/ROUGE): {metrics}")

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:186: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

...

🚀 ফাইন-টুনিং শুরু হচ্ছে...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


✅ সম্পন্ন হয়েছে! এআই উত্তর: আমি বুঝতে পারি যে আপনার জন্য এটা খুব কঠিন হচ্ছে। এটা
📊 মেট্রিক্স (BLEU/ROUGE): {'bleu': 0.0, 'rougeL': np.float64(0.0)}


In [9]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("assessment_results.db")
print("LLAMAExperiments Table:")
display(pd.read_sql_query("SELECT * FROM LLAMAExperiments", conn))
print("\nGeneratedResponses Table:")
display(pd.read_sql_query("SELECT * FROM GeneratedResponses", conn))
conn.close()

LLAMAExperiments Table:


,id,model_name,lora_config,train_loss,metrics,timestamp
0,1,Llama-3.1-8B-bnb-4bit,None,0.914329,"{""bleu"": 0.0, ""rougeL"": 0.0}",2026-04-01T20:24:28.229355



GeneratedResponses Table:


,id,experiment_id,input_text,response_text,timestamp
0,1,1,"আমার খুব একা লাগছে, আমি কি করতে পারি?",আমি বুঝতে পারি যে আপনার জন্য এটা খুব কঠিন হচ্ছ...,2026-04-01T20:24:28.229609
